# Project 2 — Moving JetAuto in Gazebo

<svg width="100%" viewBox="0 0 600 360" xmlns="http://www.w3.org/2000/svg"><rect width="600" height="360" fill="#f8fafc"/><polyline points="120,260 420,260 420,80 120,80 120,260" fill="none" stroke="#0f172a" stroke-width="4"/><circle cx="120" cy="260" r="8" fill="#16a34a"/><text x="95" y="290" font-family="Arial" font-size="16">Start</text><text x="250" y="250" font-family="Arial" font-size="15">1 m forward</text><text x="430" y="175" font-family="Arial" font-size="15">side-left</text><text x="250" y="70" font-family="Arial" font-size="15">side-right after turn</text><path d="M120 80 Q70 170 120 260" fill="none" stroke="#64748b" stroke-dasharray="8 6" stroke-width="3"/><text x="35" y="175" font-family="Arial" font-size="15">return arc</text></svg>


## Project goal
Create a ROS node that moves the simulated JetAuto robot through a roughly 1-meter square pattern in Gazebo.

## Required pattern
After a start command:
1. Move forward about 1 m.
2. Move left about 1 m without changing heading.
3. Rotate clockwise about 90 degrees.
4. Move right about 1 m while facing inward.
5. Return to the start using either an advanced turning motion or the simpler forward-then-turn option.
6. Repeat the pattern twice.


## Suggested package setup
```bash
cd ~/ros_ws/src
catkin_create_pkg lab4_jetauto_control rospy geometry_msgs
mkdir -p lab4_jetauto_control/scripts
cd ~/ros_ws
catkin_make
source devel/setup.bash
```


## Controller design tips
Because simulation timing can drift, start with conservative velocities and measure actual distance. A simple open-loop strategy can use `speed × duration`, but a stronger solution records odometry or simulation pose to decide when each segment is complete.

Recommended structure:
- `publish_twist(x, y, yaw_rate, duration)`
- `stop()` after each segment
- `run_square_pattern(repeats=2)`
- `input()` prompt before starting


## Test commands
```bash
roslaunch jetauto_gazebo worlds.launch
rosrun lab4_jetauto_control jetauto_control.py
rostopic echo /jetauto_controller/cmd_vel
```
Record the Gazebo window while the robot completes the pattern.


## Assessment questions

### 1. What command did you use to launch JetAuto in Gazebo?

```bash
# Set environment variables for the robot variant
export MACHINE_TYPE=default
export LIDAR_TYPE=A1
export DEPTH_CAMERA_TYPE=camera

# Launch Gazebo Sim with the JetAuto robot
ros2 launch jetauto_description gazebo.launch.py
```

This launches:
- **Gazebo Sim** (`gz sim empty.sdf`) – the physics engine (server + GUI on macOS)
- **Robot State Publisher** – publishes TF frames from the URDF
- **Four `ros_gz_bridge` nodes** – bridge `/cmd_vel` (ROS→Gz), odometry (Gz→ROS),
  `/scan` (Gz→ROS), and `/imu_data` (Gz→ROS)
- **Spawner** – places the JetAuto model at (0, 0, 0.1) with default yaw = 0
  (facing +x / east in ROS REP‑103)

The environment variables select the standard (non‑Pro) JetAuto chassis with an A1
LiDAR and depth camera.

---

### 2. Which launch files are included by the main world launch file?

The main launch file `jetauto_description/launch/gazebo.launch.py` is a **Python**
launch file.  It does **not** include other launch files; instead it directly
orchestrates these components:

| Component | Type | Description |
|-----------|------|-------------|
| `xacro_to_urdf` | `ExecuteProcess` | Compiles `jetauto.urdf.xacro` → `/tmp/jetauto.urdf` |
| `gz_server` / `gz_gui` | `ExecuteProcess` | Starts `gz sim` (server + GUI on macOS; combined on Linux) |
| `robot_state_publisher` | `Node` | Publishes TF frames from the URDF |
| `bridge_cmd_vel` | `Node` (`ros_gz_bridge`) | `/cmd_vel` (Twist) → Gazebo |
| `bridge_odom` | `Node` (`ros_gz_bridge`) | Gazebo odometry → ROS `Odometry` |
| `bridge_scan` | `Node` (`ros_gz_bridge`) | Gazebo LaserScan → ROS |
| `bridge_imu` | `Node` (`ros_gz_bridge`) | Gazebo IMU → ROS |
| `spawn_robot` | `ExecuteProcess` | `ros2 run ros_gz_sim create` to spawn the URDF |

The package also contains two other launch files:
- **`display.launch.py`** — RViz2 visualisation without Gazebo
- **`control_view.launch.py`** — RViz2 with control‑view config (used alongside Gazebo)

The original ROS1 `jetauto_gazebo` package had XML launch files
(`worlds.launch`, `empty_world.launch`, `spwan_model.launch`) but these are not
compatible with ROS2 and have been replaced by the Python launch file above.

---

### 3. How did you set up the workspace and package?

**Workspace**: The ROS2 workspace at `ros2_ws/` was already initialised with
colcon and contained several existing packages (`jetauto_description`,
`py_pubsub_lab3`, etc.).  The workspace was previously built with
`colcon build`.

**Package creation steps**:

```bash
# 1. Create package directory structure
cd ros2_ws/src
mkdir -p lab4_jetauto_control/{lab4_jetauto_control,resource,launch}

# 2. Create package.xml (format 3, ament_python build type)
#    Dependencies: rclpy, geometry_msgs, nav_msgs

# 3. Create setup.py with entry point:
#    jetauto_square = lab4_jetauto_control.jetauto_square:main

# 4. Create setup.cfg for script installation

# 5. Write the control node at
#    lab4_jetauto_control/lab4_jetauto_control/jetauto_square.py

# 6. Build
cd ../..
colcon build --packages-select lab4_jetauto_control

# 7. Source the overlay
source install/setup.bash
```

The package uses `ament_python` (the standard ROS2 Python build type) with a
`setuptools`‑based `setup.py`.  The entry point is registered as a console
script so the node can be launched with `ros2 run lab4_jetauto_control
jetauto_square`.

---

### 4. If you implemented the advanced turning option, what made curved motion difficult and how did you tune it?

I implemented the **simpler forward‑then‑turn** return strategy for reliability.
The robot at (0, 1) heading south simply drives forward 1 m (returning to
(0, 0), the start) and then turns CCW 90° to restore its original east‑facing
heading.

**What would make a true curved‑return (arc) difficult:**

1. **Coordinated linear + angular velocity**:  A circular arc requires a fixed
   ratio `v / ω = radius`.  The path from (0, 1) facing south back to (0, 0)
   facing east is not a simple circular arc — the instantaneous curvature
   changes along the path.

2. **Simulation timing drift**:  Even small integration errors in Gazebo cause
   the robot to overshoot or undershoot the target.  A purely open‑loop arc
   (`publish twist` for fixed duration) would miss the start by 10–30 cm.

3. **Odometry‑feedback on arcs**:  Computing the `remaining arc length` and
   `tangent angle` from current pose requires solving the inverse problem
   (given a target pose, what arc gets you there?), which is non‑trivial for
   arbitrary start–target pairs.

4. **Tuning**:  You would need to:
   - Compute the instantaneous curvature `κ = ω / v`
   - Continuously re‑plan the arc as odometry updates
   - Tune both linear and angular P‑gains simultaneously (they interact)
   - Add a terminal‑approach phase to avoid oscillation near the target

**Why the simple option works well**:  The forward‑then‑turn strategy separates
the linear and angular errors, so each P‑controller only has one degree of
freedom to manage.  The per‑step odometry reset ensures that any small drift in
one segment does not affect the next.

## Solution — ROS2 Implementation

This notebook assumes the robostack‑jazzy conda environment and the `ros2_ws` workspace
alongside this notebook.  All commands are run from the workspace root.

### 1. Package creation & workspace setup

In [ ]:
# Step 1: Create the ROS2 Python package structure
# (Run once — this has already been done for the solution)

import os

ws_src = '../ros2_ws/src'
pkg_name = 'lab4_jetauto_control'
pkg_dir = os.path.join(ws_src, pkg_name)

# Create directory structure
os.makedirs(os.path.join(pkg_dir, pkg_name), exist_ok=True)
os.makedirs(os.path.join(pkg_dir, 'resource'), exist_ok=True)
os.makedirs(os.path.join(pkg_dir, 'launch'), exist_ok=True)

print(f'Package directory structure created at: {pkg_dir}')
print('Files:')
for root, dirs, files in os.walk(pkg_dir):
    for f in files:
        print(f'  {os.path.relpath(os.path.join(root, f), pkg_dir)}')

In [ ]:
# Step 2: Build the package with colcon
# Run this in a terminal with the ROS2 conda environment:
#
#   cd ../ros2_ws
#   conda activate ros2
#   source install/setup.bash
#   colcon build --packages-select lab4_jetauto_control
#
# Then verify:
#   ros2 pkg list | grep lab4
#   ros2 pkg executables lab4_jetauto_control

### 2. Control node — `jetauto_square.py`

The complete source is at [`ros2_ws/src/lab4_jetauto_control/lab4_jetauto_control/jetauto_square.py`](ros2_ws/src/lab4_jetauto_control/lab4_jetauto_control/jetauto_square.py).

**Design overview**

| Aspect | Detail |
|--------|--------|
| Control | Odometry‑feedback; P‑controller on distance & heading |
| Pattern | State machine with 10 relative steps per repetition |
| Lateral moves | Turn‑move‑turn (diff‑drive can't strafe) |
| Return to start | Forward 1 m + turn to original heading (simple option) |
| Repetitions | 2 full squares |

**Coordinate note:** The robot spawns at yaw = 0 (facing +x / **east** in ROS
REP‑103).  All steps are *relative* to the robot's current pose, so the square
is always 1 m × 1 m regardless of spawn orientation.  The robot returns to its
exact start position with its original heading.

**Step trace (one repetition, yaw = 0 = facing east / +x)**

| # | Action | Yaw | Position |
|---|--------|-----|----------|
| 0 | Start | 0° (E) | (0, 0) |
| 1 | Forward 1 m | 0° (E) | (1, 0) |
| 2 | Turn CCW 90° (prep side‑left) | +90° (N) | (1, 0) |
| 3 | Forward 1 m (side‑left) | +90° (N) | (1, 1) |
| 4 | Turn CW 90° (restore heading) | 0° (E) | (1, 1) |
| 5 | Turn CW 90° (rotate clockwise) | −90° (S) | (1, 1) |
| 6 | Turn CW 90° (prep right/inward) | ±180° (W) | (1, 1) |
| 7 | Forward 1 m (right / inward) | ±180° (W) | (0, 1) |
| 8 | Turn CCW 90° (align for return) | −90° (S) | (0, 1) |
| 9 | **Forward 1 m** (return!) | −90° (S) | **(0, 0)** |
| 10 | Turn CCW 90° (restore) | 0° (E) | (0, 0) |

Each step uses odometry feedback to decide when the target distance or angle
is reached, so drift does not accumulate across repetitions.

### 3. How to run the complete demo

In [ ]:
# ─── Terminal 1 — Launch Gazebo + JetAuto ────────────────────────
cd ../ros2_ws
conda activate ros2
source install/setup.bash
export MACHINE_TYPE=default
export LIDAR_TYPE=A1
export DEPTH_CAMERA_TYPE=camera
ros2 launch jetauto_description gazebo.launch.py

# ─── Terminal 2 — Run the square controller ──────────────────────
cd ../ros2_ws
conda activate ros2
source install/setup.bash
ros2 run lab4_jetauto_control jetauto_square
# (press ENTER when prompted)

# ─── Terminal 3 (optional) — Monitor velocity commands ───────────
conda activate ros2
source ../ros2_ws/install/setup.bash
ros2 topic echo /cmd_vel

# ─── Terminal 4 (optional) — Monitor odometry ────────────────────
conda activate ros2
source ../ros2_ws/install/setup.bash
ros2 topic echo /model/jetauto/odometry